In [1]:
from pyspark.sql import SparkSession
from pyspark.streaming import StreamingContext
import os
from pyspark.sql.functions import max,min
import logging

In [2]:
# definaion of the variable for kafka
topic_name = "debezium.commerce.products"
bootstrap_server = "localhost:9093"
spark_version= "3.5.2"
scala_version = "2.12"

In [3]:
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version},org.apache.hadoop:hadoop-aws:3.3.1,com.amazonaws:aws-java-sdk-bundle:1.11.874 pyspark-shell'
spark = SparkSession.builder\
   .master("local")\
   .config("spark.driver.port", "4050") \
   .config("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.5.2,com.amazonaws:aws-java-sdk-bundle:1.11.874,org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1") \
   .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
   .config("spark.hadoop.fs.s3a.access.key", "minio") \
   .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
   .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
   .config("spark.hadoop.fs.s3a.path.style.access", "true") \
   .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")\
   .appName("kafka-streaming")\
   .getOrCreate()
# set logger level with format
logging.basicConfig(format='%(asctime)s %(levelname)-8s %(message)s', level=logging.INFO, datefmt='%Y-%m-%d %H:%M:%S')

25/01/12 22:27:28 WARN Utils: Your hostname, depq-2.local resolves to a loopback address: 127.0.0.1; using 192.168.1.6 instead (on interface en0)
25/01/12 22:27:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/lap14646/Library/Python/3.9/lib/python/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/lap14646/.ivy2/cache
The jars for the packages stored in: /Users/lap14646/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-db6f596f-dee3-4d34-9757-6cb54a016f2f;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.2 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in local-m2-cache
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in local-m2-cache
	found org.apache.hadoop#hadoop-client-api;3.3.4 in local-m2-cache
	found commons-logging#commons-logging;1.1.3 in local-m2-cache
	found com.google.code.findbugs#jsr

OSError: [Errno 49] Can't assign requested address

In [24]:
spark

2025-01-12 01:08:05 INFO     Closing down clientserver connection


ConnectionRefusedError: [Errno 61] Connection refused

In [8]:
def read_df_from_kafka(topic_name, bootstrap_server):
    try:
        df_cdc = spark.readStream.format("kafka") \
            .option("kafka.bootstrap.servers", bootstrap_server) \
            .option("subscribe", topic_name) \
            .option("startingOffsets", "earliest") \
            .option("kafka.security.protocol", "PLAINTEXT") \
            .load()
        logging.info("Reading from Kafka successfully!")
    except Exception as e:
        logging.error(f"Error reading from Kafka: {e}")
    return df_cdc

In [9]:
from pyspark.sql.functions import from_json, col, expr
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
# create structype for schema
product_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("price", FloatType(), True)
])
data_schema = StructType([
        StructField("payload", StructType([
            StructField("after", product_schema, True)
        ]), True)
    ])

In [10]:
data_schema

StructType([StructField('payload', StructType([StructField('after', StructType([StructField('id', IntegerType(), True), StructField('name', StringType(), True), StructField('description', StringType(), True), StructField('price', FloatType(), True)]), True)]), True)])

In [11]:
from pyspark.sql.functions import from_json, col, expr
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
# Extract value from the Kafka DataFrame (Kafka messages are typically serialized as bytes)
df_cdc = read_df_from_kafka(topic_name, bootstrap_server)

ERROR:root:Error reading from Kafka: name 'spark' is not defined


UnboundLocalError: local variable 'df_cdc' referenced before assignment

In [12]:
df_cdc.printSchema()

NameError: name 'df_cdc' is not defined

In [10]:
# transformation cdc debezium format
parsed_df = df_cdc.selectExpr("CAST(value AS STRING) as json") \
.select(from_json(col("json"), data_schema).alias("data")) \
.select("data.payload.after.*")

In [11]:
df_cdc.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [12]:
# Version 2 
from pyspark.sql.functions import from_json, col, expr
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
# create structype for schema
# Define JSON schema
product_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("price", FloatType(), True)
])

# Define JSON schema
json_schema = StructType([
    StructField("op", StringType(), True),
    StructField("payload", StructType([
        StructField("before", product_schema, True),
        StructField("after", product_schema, True),
    ]), True)
])

# Filter non-null values and parse JSON data
new_df = df_cdc.filter("value is not null") \
    .withColumn("value", from_json(col("value").cast("string"), json_schema)) \
    .withColumn("cdc_data", expr("case when value.op = 'd' then value.payload.before else value.payload.after end")) \
    .withColumn("is_delete", expr("case when value.op = 'd' then 1 else 0 end")) \
    .withColumn("ver", expr("cast(partition as long) * 100000000000000000 + offset")) \
    .select("cdc_data.*", "is_delete", "ver", "value.op", "timestamp")


In [13]:
new_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- price: float (nullable = true)
 |-- is_delete: integer (nullable = false)
 |-- ver: long (nullable = true)
 |-- op: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [14]:
# how to write to s3 minio
new_df.writeStream.format("parquet") \
    .option("path", "s3a://raw/") \
    .option("checkpointLocation", "s3a://raw/checkpoint") \
    .outputMode("append") \
    .trigger(processingTime="60 seconds") \
    .start() \
    .awaitTermination()

25/01/10 14:24:52 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/01/10 14:24:52 WARN BasicProfileConfigLoader: Your profile name includes a 'profile ' prefix. This is considered part of the profile name in the Java SDK, so you will need to include this prefix in your profile name when you reference this profile from your Java code.
25/01/10 14:24:52 WARN BasicProfileConfigLoader: Your profile name includes a 'profile ' prefix. This is considered part of the profile name in the Java SDK, so you will need to include this prefix in your profile name when you reference this profile from your Java code.
25/01/10 14:24:52 WARN BasicProfileConfigLoader: Your profile name includes a 'profile ' prefix. This is considered part of the profile name in the Java SDK, so you will need to include this prefix in your profile name when you reference this profile from your Java code.
25/01/10 14:24:52 WARN BasicProfileConfig

KeyboardInterrupt: 

25/01/10 14:52:41 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1001 (kafka/127.0.0.1:9093) could not be established. Broker may not be available.
25/01/10 14:52:42 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1001 (kafka/127.0.0.1:9093) could not be established. Broker may not be available.
25/01/10 14:52:43 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1001 (kafka/127.0.0.1:9093) could not be established. Broker may not be available.
25/01/10 14:52:44 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1001 (kafka/127.0.0.1:9093) could not be established. Broker may not be available.
25/01/10 14:52:45 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1001 (kafka/127.0.0.1:9093) could not be established. Broker may not be available.
25/01/10 14:52:46 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1001 (kafka/127.0.0.1

In [1]:
# create table in iceberg
spark.sql("""CREATE TABLE IF NOT EXISTS mart.products (
            id INT,
            name STRING,
            description STRING,
            price FLOAT
        )
          USING iceberg 
          Location 's3a://raw/products'
          partitioned by (id)""")

NameError: name 'parsed_df' is not defined